In [3]:
import pandas as pd
from pathlib import Path
import xarray as xr

data_dir = Path("/Users/jakobwerkgarner/code/mt_dsnow/par_sens/SNOWPACK_data_seasons_daily")

data_dir_out = Path("/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data")

nc_path = data_dir / "SNOWPACK_data_seasons_daily.nc"
snowpack_ds = xr.open_dataset(nc_path)

snowpack_ds

<xarray.Dataset>
Dimensions:  (time: 363, station: 8)
Coordinates:
  * time     (time) datetime64[ns] 2023-11-01 2023-11-02 ... 2025-04-30
  * station  (station) object 'AXLIZ1' 'LADU2' 'MTIM2' ... 'SMAD2' 'SROT2'
Data variables:
    season   (time, station) object ...
    HS_mod   (time, station) float64 ...
    HS_meas  (time, station) float64 ...
    SWE      (time, station) float64 ...

In [52]:
train_season = seasons[0]
val_season = seasons[1]

train_ds = snowpack_ds.where(snowpack_ds["season"] == train_season, drop=True)
val_ds = snowpack_ds.where(snowpack_ds["season"] == val_season, drop=True)

train_out = data_dir_out / f"allstations_{train_season}_train.nc"
val_out = data_dir_out / f"allstations_{val_season}_validation.nc"

train_ds.to_netcdf(train_out)
val_ds.to_netcdf(val_out)

print(f"Saved train_ds -> {train_out}")
print(f"Saved val_ds   -> {val_out}")

train_ds, val_ds

Saved train_ds -> /Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data/allstations_2023_2024_train.nc
Saved val_ds   -> /Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data/allstations_2024_2025_validation.nc


(<xarray.Dataset>
 Dimensions:  (time: 182, station: 8)
 Coordinates:
   * time     (time) datetime64[ns] 2023-11-01 2023-11-02 ... 2024-04-30
   * station  (station) object 'AXLIZ1' 'LADU2' 'MTIM2' ... 'SMAD2' 'SROT2'
 Data variables:
     season   (time, station) object '2023_2024' '2023_2024' ... '2023_2024'
     HS_mod   (time, station) float64 0.1877 0.1101 0.182 ... 1.39 2.14 0.5188
     HS_meas  (time, station) float64 0.209 0.113 0.1752 ... 1.403 2.116 0.4868
     SWE      (time, station) float64 29.08 13.99 31.94 nan ... 678.8 808.8 250.3,
 <xarray.Dataset>
 Dimensions:  (time: 181, station: 8)
 Coordinates:
   * time     (time) datetime64[ns] 2024-11-01 2024-11-02 ... 2025-04-30
   * station  (station) object 'AXLIZ1' 'LADU2' 'MTIM2' ... 'SMAD2' 'SROT2'
 Data variables:
     season   (time, station) object '2024_2025' '2024_2025' ... '2024_2025' nan
     HS_mod   (time, station) float64 0.0 0.0 0.0 0.0 ... 0.0 0.002955 0.9362 nan
     HS_meas  (time, station) float64 0.0 0.0 

# Ok try Wikler21

In [5]:
train_2023_2024_path = Path("/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_data/raw_data/dsnow/Win21_calib/Win21_all.nc")



In [7]:
# Split Win21 data into alternating winter seasons (validation/train), station-wise,
# following the same logic as in the R script.

def split_alternating_winters(ds, hs_var="HS", swe_var="SWE"):
    df = ds[[hs_var, swe_var]].to_dataframe().reset_index()
    df = df.sort_values(["station", "time"])

    train_parts = []
    val_parts = []

    for station, d in df.groupby("station"):
        # mimic observed-series behavior: keep rows with HS observations
        d = d.dropna(subset=[hs_var]).sort_values("time").copy()
        if d.empty:
            continue

        years = sorted(d["time"].dt.year.unique())
        idx_y = 1  # first accepted winter -> validation, second -> training, etc.

        for y in years[:-1]:
            start = pd.Timestamp(year=y, month=8, day=1)
            end = pd.Timestamp(year=y + 1, month=7, day=31)

            winter = d[(d["time"] >= start) & (d["time"] <= end)].copy()
            n = len(winter)

            if n < 200:
                continue

            first_hs = winter[hs_var].iloc[0]
            last_hs = winter[hs_var].iloc[-1]

            if n < 365 and ((first_hs > 0.05) or (last_hs > 0.05)):
                continue

            winter["season"] = f"{y}_{y+1}"

            if idx_y % 2 == 0:
                train_parts.append(winter)
            else:
                val_parts.append(winter)

            idx_y += 1

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=df.columns.tolist() + ["season"])
    val_df = pd.concat(val_parts, ignore_index=True) if val_parts else pd.DataFrame(columns=df.columns.tolist() + ["season"])

    train_ds_split = train_df.set_index(["station", "time"]).to_xarray() if not train_df.empty else xr.Dataset()
    val_ds_split = val_df.set_index(["station", "time"]).to_xarray() if not val_df.empty else xr.Dataset()

    return train_ds_split, val_ds_split


train_2023_2024_train_ds, train_2023_2024_val_ds = split_alternating_winters(train_2023_2024_ds)

train_2023_2024_train_path = data_dir_out / "Win21_all_train.nc"
train_2023_2024_val_path = data_dir_out / "Win21_all_validation.nc"

train_2023_2024_train_ds.to_netcdf(train_2023_2024_train_path)
train_2023_2024_val_ds.to_netcdf(train_2023_2024_val_path)

print(f"Saved train -> {train_2023_2024_train_path}")
print(f"Saved val   -> {train_2023_2024_val_path}")
train_2023_2024_train_ds, train_2023_2024_val_ds

Saved train -> /Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data/Win21_all_train.nc
Saved val   -> /Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data/Win21_all_validation.nc


(<xarray.Dataset>
 Dimensions:  (station: 15, time: 6880)
 Coordinates:
   * station  (station) object 'Davos_Flueelastr' ... 'Zuoz'
   * time     (time) datetime64[ns] 1961-08-01 1961-08-02 ... 2022-07-31
 Data variables:
     HS       (station, time) float64 nan nan nan nan nan ... 0.0 0.0 0.0 0.0 0.0
     SWE      (station, time) float64 nan nan nan nan nan ... nan nan nan nan nan
     season   (station, time) object nan nan nan ... '2021_2022' '2021_2022',
 <xarray.Dataset>
 Dimensions:  (station: 15, time: 7637)
 Coordinates:
   * station  (station) object 'Davos_Flueelastr' ... 'Zuoz'
   * time     (time) datetime64[ns] 1960-08-01 1960-08-02 ... 2021-07-31
 Data variables:
     HS       (station, time) float64 nan nan nan nan nan ... 0.0 0.0 0.0 0.0 0.0
     SWE      (station, time) float64 nan nan nan nan nan ... nan nan nan nan nan
     season   (station, time) object nan nan nan ... '2020_2021' '2020_2021')